# TrendHive — Dubai Cafes ML + Analytics (Random Forest + Dashboard + Map)

This notebook trains a **Random Forest** model to predict a **Popularity Tier** for cafes, builds **area-level analytics**, creates an **opportunity ranking**, and renders **map clustering / heat**.

## What you need to upload to Colab
Upload these files into the Colab session (Files panel) and set their paths below:

- `dubai_cafes_last4combined.xlsx`
- `dubai_cafes_50plus_reviews_ALL_REVIEWS.xlsx`

> If your column names differ slightly, adjust the `COLMAP` section.


In [ ]:
# Install dependencies
!pip -q install openpyxl textblob scikit-learn folium joblib tqdm

import numpy as np
import pandas as pd
from pathlib import Path
from textblob import TextBlob
from tqdm import tqdm
import joblib


In [ ]:
# ==== Set your file paths here ====
# For Colab: use "/content/..." paths after uploading files
# For local: use relative paths to the datasets/ folder
from pathlib import Path

# Auto-detect: Colab vs local
_BASE = Path("/content") if Path("/content").exists() else Path(__file__).resolve().parent.parent if "__file__" in dir() else Path(".").resolve().parent

CAFES_XLSX = str(_BASE / "datasets" / "dubai_cafes_last4combined.xlsx")
REVIEWS_XLSX = str(_BASE / "datasets" / "dubai_cafes_50plus_reviews_ALL_REVIEWS.xlsx")

# Fallback to Colab paths if local doesn't exist
if not Path(CAFES_XLSX).exists():
    CAFES_XLSX = "/content/dubai_cafes_last4combined.xlsx"
if not Path(REVIEWS_XLSX).exists():
    REVIEWS_XLSX = "/content/dubai_cafes_50plus_reviews_ALL_REVIEWS.xlsx"

assert Path(CAFES_XLSX).exists(), f"Missing: {CAFES_XLSX}"
assert Path(REVIEWS_XLSX).exists(), f"Missing: {REVIEWS_XLSX}"

print(f"Cafes file:   {CAFES_XLSX}")
print(f"Reviews file: {REVIEWS_XLSX}")

## 1) Load data + (optional) column mapping

If your Excel columns are named differently, update the mapping below.


In [ ]:
# --- Optional: column mapping (only edit if needed) ---
# If your columns already match, leave as-is.
COLMAP_CAFES = {
    "Place_ID": "Place_ID",
    "Name": "Name",
    "Rating": "Rating",
    "Reviews": "Reviews",
    "Latitude": "Latitude",
    "Longitude": "Longitude",
    "detected_area": "detected_area",
    "rent_tier": "rent_tier",
    "competition_level": "competition_level",
    "utility_level": "utility_level",
    "utility_cost_aed_month": "utility_cost_aed_month",
    "avg_commercial_rent_aed_sqft_year": "avg_commercial_rent_aed_sqft_year",
    "competitors_within_300m": "competitors_within_300m",
    "competitors_within_500m": "competitors_within_500m",
    "competitors_within_1000m": "competitors_within_1000m",
    "free_parking_lot": "free_parking_lot",
    "paid_parking_lot": "paid_parking_lot",
    "free_street_parking": "free_street_parking",
    "paid_street_parking": "paid_street_parking",
    "valet_parking": "valet_parking",
}

COLMAP_REVIEWS = {
    "Place_ID": "Place_ID",
    "Review_Text": "Review_Text",
    "Review_Rating": "Review_Rating",
}

def load_cafes(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    # rename only the keys we have
    rename = {v: k for k,v in COLMAP_CAFES.items() if v in df.columns}
    df = df.rename(columns=rename)
    return df

def load_reviews(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    rename = {v: k for k,v in COLMAP_REVIEWS.items() if v in df.columns}
    df = df.rename(columns=rename)
    return df

cafes = load_cafes(CAFES_XLSX)
reviews = load_reviews(REVIEWS_XLSX)

print("cafes:", cafes.shape)
print("reviews:", reviews.shape)
display(cafes.head(3))
display(reviews.head(3))


## 2) Precompute sentiment aggregates per cafe (fast to reuse)

We compute:
- `avg_review_rating`
- `avg_polarity`
- `neg_share` (polarity < -0.1)
- `pos_share` (polarity > 0.1)

Then save a cached CSV so future runs are fast.


In [ ]:
def polarity(txt: str) -> float:
    try:
        return TextBlob(str(txt)).sentiment.polarity
    except Exception:
        return np.nan

# Cache file: Colab or local
if Path("/content").exists():
    CACHE_AGG = "/content/review_agg_by_place.csv"
else:
    CACHE_AGG = str(Path(CAFES_XLSX).parent.parent / "ml" / "review_agg_by_place.csv")

if Path(CACHE_AGG).exists():
    agg = pd.read_csv(CACHE_AGG)
    print("Loaded cached review aggregates", agg.shape)
else:
    r = reviews.dropna(subset=["Review_Text"]).copy()
    # compute polarity with progress bar
    tqdm.pandas()
    r["polarity"] = r["Review_Text"].astype(str).progress_map(polarity)
    r["is_neg"] = (r["polarity"] < -0.10).astype(int)
    r["is_pos"] = (r["polarity"] > 0.10).astype(int)

    agg = r.groupby("Place_ID").agg(
        avg_review_rating=("Review_Rating", "mean"),
        avg_polarity=("polarity", "mean"),
        neg_share=("is_neg", "mean"),
        pos_share=("is_pos", "mean"),
        reviews_used=("Review_Text", "count"),
    ).reset_index()

    agg.to_csv(CACHE_AGG, index=False)
    print("Computed + cached review aggregates", agg.shape)

display(agg.head())

## 3) Build training table + proxy target (Popularity Tier)

**popularity_score = log(1 + Reviews) * Rating**

Then split into 5 classes (quantiles):
- Very Low / Low / Medium / High / Very High


In [ ]:
df = cafes.merge(agg, on="Place_ID", how="left")

df["Rating"] = pd.to_numeric(df.get("Rating"), errors="coerce")
df["Reviews"] = pd.to_numeric(df.get("Reviews"), errors="coerce")

df["popularity_score"] = np.log1p(df["Reviews"].fillna(0)) * df["Rating"].fillna(0)

q = df["popularity_score"].quantile([0.2, 0.4, 0.6, 0.8]).to_dict()
bins = [-np.inf, q[0.2], q[0.4], q[0.6], q[0.8], np.inf]
labels = ["Very Low", "Low", "Medium", "High", "Very High"]
df["popularity_tier"] = pd.cut(df["popularity_score"], bins=bins, labels=labels)

print(df["popularity_tier"].value_counts(dropna=False))
display(df[["Name","Rating","Reviews","avg_polarity","popularity_score","popularity_tier"]].head(5))


## 4) Train Random Forest classifier

Features include:
- rating, reviews
- competitor counts
- rent + utility cost
- parking flags
- sentiment aggregates
- categorical tiers/area (one-hot)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

FEATURE_COLS = [
    "Rating", "Reviews",
    "free_parking_lot", "paid_parking_lot", "free_street_parking", "paid_street_parking", "valet_parking",
    "utility_cost_aed_month", "avg_commercial_rent_aed_sqft_year",
    "competitors_within_300m", "competitors_within_500m", "competitors_within_1000m",
    "avg_review_rating", "avg_polarity", "neg_share", "pos_share",
    "utility_level", "rent_tier", "competition_level", "detected_area"
]

# Ensure missing columns exist
for c in FEATURE_COLS:
    if c not in df.columns:
        df[c] = np.nan

X = df[FEATURE_COLS].copy()
y = df["popularity_tier"].astype(str)

num_cols = [c for c in FEATURE_COLS if X[c].dtype != "object"]
cat_cols = [c for c in FEATURE_COLS if X[c].dtype == "object"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
])

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipe = Pipeline([("pre", pre), ("rf", rf)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

acc = accuracy_score(y_test, pred)
print("Accuracy:", round(acc, 4))
print(classification_report(y_test, pred))

# Save model: Colab or local
if Path("/content").exists():
    MODEL_FILE = "/content/popularity_rf.joblib"
else:
    MODEL_FILE = str(Path(CAFES_XLSX).parent.parent / "backend" / "models" / "popularity_rf.joblib")

joblib.dump(pipe, MODEL_FILE)
print("Saved model ->", MODEL_FILE)

# Also save feature list and classes
import json
_model_dir = Path(MODEL_FILE).parent
with open(_model_dir / "popularity_rf_features.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
with open(_model_dir / "popularity_rf_classes.json", "w") as f:
    json.dump(list(pipe.named_steps["rf"].classes_), f, indent=2)
print("Saved feature list + class labels")

## 5) Predict tiers for all cafes + attach to dataset

In [ ]:
model = joblib.load(MODEL_FILE)

df["predicted_tier"] = model.predict(df[FEATURE_COLS])
proba = model.predict_proba(df[FEATURE_COLS])

classes = list(model.named_steps["rf"].classes_)
df["prediction_confidence"] = proba.max(axis=1)

display(df[["Name","detected_area","Rating","Reviews","avg_polarity","predicted_tier","prediction_confidence"]].head(10))


## 6) Business Dashboard Analytics (Area summary + Opportunity ranking)

In [ ]:
def overview_cards(d: pd.DataFrame) -> dict:
    return {
        "total_cafes": int(d["Place_ID"].nunique()),
        "avg_rating": float(d["Rating"].mean()),
        "avg_reviews": float(d["Reviews"].mean()),
        "top_area_by_count": str(d["detected_area"].value_counts().index[0]) if d["detected_area"].notna().any() else None,
        "share_high_or_very_high": float((d["predicted_tier"].isin(["High","Very High"])).mean()),
    }

cards = overview_cards(df)
cards


In [ ]:
area = df.groupby("detected_area").agg(
    cafes=("Place_ID","count"),
    avg_rating=("Rating","mean"),
    avg_reviews=("Reviews","mean"),
    avg_sentiment=("avg_polarity","mean"),
    avg_comp_500=("competitors_within_500m","mean"),
    avg_rent=("avg_commercial_rent_aed_sqft_year","mean"),
    avg_utility=("utility_cost_aed_month","mean"),
).reset_index().sort_values("cafes", ascending=False)

display(area.head(15))


In [ ]:
# Opportunity ranking: high rating/sentiment/reviews, low competition/rent/utility

tmp = df.copy()
for col in ["avg_polarity","Rating","Reviews","competitors_within_500m","avg_commercial_rent_aed_sqft_year","utility_cost_aed_month"]:
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

a = tmp.groupby("detected_area").agg(
    cafes=("Place_ID","count"),
    rating=("Rating","mean"),
    sentiment=("avg_polarity","mean"),
    reviews=("Reviews","mean"),
    comp=("competitors_within_500m","mean"),
    rent=("avg_commercial_rent_aed_sqft_year","mean"),
    utility=("utility_cost_aed_month","mean"),
).reset_index()

def z(s):
    s = s.fillna(s.median())
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

a["opportunity_score"] = (
    0.35*z(a["rating"]) +
    0.25*z(a["sentiment"]) +
    0.15*z(a["reviews"]) -
    0.15*z(a["comp"]) -
    0.07*z(a["rent"]) -
    0.03*z(a["utility"])
)

opps = a.sort_values("opportunity_score", ascending=False)
display(opps.head(20))


## 7) Map Dashboard (Markers + Grid Heatmap)

We generate:
- **Marker cluster map** (popups show rating/reviews/sentiment/predicted tier)
- **Heat grid cells** (precision=3 ~ ~100m clusters)

You can switch heat metric to:
- `cafes` (density)
- `avg_rating`
- `share_high` (High or Very High predicted tiers)


In [ ]:
import folium
from folium.plugins import MarkerCluster


In [ ]:
# Marker cluster map (sample a bit if huge)
m_df = df.dropna(subset=["Latitude","Longitude"]).copy()

# if dataset very large, sample
MAX_MARKERS = 3000
if len(m_df) > MAX_MARKERS:
    m_df = m_df.sample(MAX_MARKERS, random_state=42)

center = [m_df["Latitude"].mean(), m_df["Longitude"].mean()]
m = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")
cluster = MarkerCluster().add_to(m)

def fmt(x):
    return "" if pd.isna(x) else str(round(float(x), 3))

for _, r in m_df.iterrows():
    popup = folium.Popup(
        f"""<b>{r.get('Name','')}</b><br>
        Area: {r.get('detected_area','')}<br>
        Rating: {fmt(r.get('Rating'))} | Reviews: {fmt(r.get('Reviews'))}<br>
        Sentiment: {fmt(r.get('avg_polarity'))}<br>
        Pred Tier: <b>{r.get('predicted_tier','')}</b> (conf {fmt(r.get('prediction_confidence'))})
        """, max_width=300
    )
    folium.Marker([r["Latitude"], r["Longitude"]], popup=popup).add_to(cluster)

m


In [ ]:
# Grid heatmap cells
precision = 3
g = df.dropna(subset=["Latitude","Longitude"]).copy()
g["lat_cell"] = g["Latitude"].round(precision)
g["lng_cell"] = g["Longitude"].round(precision)
g["cell_id"] = g["lat_cell"].astype(str) + "," + g["lng_cell"].astype(str)

cell = g.groupby(["cell_id","lat_cell","lng_cell"]).agg(
    cafes=("Place_ID","count"),
    avg_rating=("Rating","mean"),
    share_high=("predicted_tier", lambda s: float(pd.Series(s).isin(["High","Very High"]).mean())),
    avg_sentiment=("avg_polarity","mean"),
).reset_index()

display(cell.head())
print("Cells:", cell.shape[0])


In [ ]:
# Heatmap-style cell markers (circle radius based on cafes)
# Choose metric: 'cafes' / 'avg_rating' / 'share_high'
METRIC = "cafes"

h = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")

# normalize for radius scaling
vals = cell[METRIC].fillna(0).astype(float)
vmin, vmax = float(vals.min()), float(vals.max())
vmax = vmax if vmax > vmin else vmin + 1.0

def scale(v):
    # radius 3..18
    return 3 + 15 * ((float(v) - vmin) / (vmax - vmin))

for _, r in cell.iterrows():
    popup = folium.Popup(
        f"""Cell: {r['cell_id']}<br>
        Cafes: {int(r['cafes'])}<br>
        Avg rating: {round(float(r['avg_rating']),3) if pd.notna(r['avg_rating']) else ''}<br>
        Share High: {round(float(r['share_high']),3) if pd.notna(r['share_high']) else ''}<br>
        Avg sentiment: {round(float(r['avg_sentiment']),3) if pd.notna(r['avg_sentiment']) else ''}<br>
        """, max_width=280
    )
    folium.CircleMarker(
        location=[r["lat_cell"], r["lng_cell"]],
        radius=scale(r[METRIC]),
        fill=True,
        popup=popup
    ).add_to(h)

h


## 8) Export outputs (optional)

This saves:
- predictions CSV
- area summary CSV
- opportunities CSV
- grid cells CSV


In [ ]:
# Save outputs: Colab or local
if Path("/content").exists():
    OUT_DIR = Path("/content/outputs")
else:
    OUT_DIR = Path(CAFES_XLSX).parent.parent / "backend" / "data" / "outputs"

OUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUT_DIR/"cafes_with_predictions.csv", index=False)
area.to_csv(OUT_DIR/"area_summary.csv", index=False)
opps.to_csv(OUT_DIR/"opportunity_ranking.csv", index=False)
cell.to_csv(OUT_DIR/"grid_cells.csv", index=False)

print("Saved to:", OUT_DIR)
for f in OUT_DIR.iterdir():
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")